# 04 - Vector Database Ingestion

Ingests chunked PubMed documents into multiple vector databases for comparison.
Each vector DB stores the same corpus with the same embeddings, enabling fair retrieval comparison.

**Supported Vector DBs:** ChromaDB, Qdrant, LanceDB, Weaviate  
**Input:** LangChain Documents from notebook 03  
**Output:** Persisted vector stores in `vectorstores/`

In [ ]:
import sys
sys.path.append("..")

from langchain_huggingface import HuggingFaceEmbeddings
from src.data_loader import load_pubmedqa, build_mesh_lookup
from src.chunking import create_documents
from src.ingestion import ingest_to_chroma, ingest_to_qdrant, ingest_to_lancedb
import config

## Prepare Documents & Embeddings

In [ ]:
df = load_pubmedqa()
mesh_lookup = build_mesh_lookup(df)
documents, ids = create_documents(str(config.DATA_RAW_DIR), mesh_lookup)

embedding_key = config.DEFAULT_EMBEDDING
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[embedding_key])
print(f"Using embedding model: {config.EMBEDDING_MODELS[embedding_key]}")

## Ingest into ChromaDB

In [ ]:
chroma_store = ingest_to_chroma(
    documents, ids, embeddings,
    db_name=f"{embedding_key}_pubmed_chroma"
)

## Ingest into Qdrant

In [ ]:
qdrant_store = ingest_to_qdrant(
    documents, ids, embeddings,
    collection_name=f"{embedding_key}_pubmed_qdrant"
)

## Ingest into LanceDB

In [ ]:
lance_db, lance_table = ingest_to_lancedb(
    documents, ids, embeddings,
    table_name=f"{embedding_key}_pubmed_lance"
)

## (Optional) Ingest into Weaviate Cloud

Requires `WEAVIATE_URL` and `WEAVIATE_API_KEY` in `.env`.

In [ ]:
# from src.ingestion import ingest_to_weaviate
# weaviate_store, weaviate_client = ingest_to_weaviate(documents, ids, embeddings)

## Verify Ingestion Counts

In [ ]:
print(f"ChromaDB: {chroma_store._collection.count()} documents")
print(f"LanceDB:  {lance_table.count_rows()} documents")
print(f"\nAll stores ingested from {len(documents)} source chunks")